# Warehouse QA: pilot Parquet tables

Checks the tables written by `load_core_tables.py` and `clean_wq_results.py` for one case study, before building on them:

1. Files and row counts
2. Integrity checks (keys and cross-table links; every count should be 0)
3. Dams, stations and links
4. Map of dams, stations, streams and HUC8 boundaries
5. `wq_measurement`: parameters, censoring, unit assumptions, value distributions
6. Raw vs cleaned row counts
7. Before/after coverage per dam (a prototype of the step 4 DuckDB views), compared with `report_pilot_eligibility.py`
8. Spot check: one dam's time series around its removal year

Spec: `docs/warehouse_spec.md`.

In [ ]:
from pathlib import Path
import sys

import duckdb
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from etl.common import wq_rules
from etl.common.config import load_config
from etl.common.warehouse import warehouse_dir

# Same case study the Slurm jobs use (data/slurm/case_study.env).
CASE_STUDY = "pilot_random5_seed42"
cfg = load_config(CASE_STUDY, project_root=ROOT)
WAREHOUSE = warehouse_dir(cfg)

window = cfg.get("wq_window", {}) or {}
YEARS_BEFORE = int(window.get("years_before", 3))
YEARS_AFTER = int(window.get("years_after", 3))

print("Case study:", cfg.name)
print("HUC8s:", cfg.huc8s)
print("Warehouse:", WAREHOUSE)
print(f"Window: {YEARS_BEFORE} years before / {YEARS_AFTER} years after removal")

## 1. Files and row counts

In [ ]:
TABLES = [
    "dam_removal", "wq_station", "dam_station_link", "huc8_watershed", "streams",
    "wq_measurement", "wq_measurement_raw",
]

con = duckdb.connect()
rows = []
for name in TABLES:
    path = WAREHOUSE / f"{name}.parquet"
    if not path.exists():
        rows.append({"table": name, "exists": False})
        continue
    # One view per table, so the queries below don't repeat file paths.
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{path}')")
    rows.append({
        "table": name,
        "exists": True,
        "rows": con.execute(f"SELECT count(*) FROM {name}").fetchone()[0],
        "size_mb": round(path.stat().st_size / 1e6, 2),
    })

pd.DataFrame(rows)

## 2. Integrity checks

Each query counts violations, so every row should show **0**. These repeat the section 4 checks the writers already ran, read back from the files on disk.

In [ ]:
CHECKS = {
    "dam_removal: duplicate AR_ID":
        "SELECT count(*) - count(DISTINCT AR_ID) FROM dam_removal",
    "wq_station: duplicate station_id":
        "SELECT count(*) - count(DISTINCT station_id) FROM wq_station",
    "dam_station_link: duplicate (AR_ID, station_id)":
        "SELECT count(*) FROM (SELECT AR_ID, station_id FROM dam_station_link GROUP BY ALL HAVING count(*) > 1)",
    "dam_station_link: AR_ID not in dam_removal":
        "SELECT count(*) FROM dam_station_link WHERE AR_ID NOT IN (SELECT AR_ID FROM dam_removal)",
    "dam_station_link: station_id not in wq_station":
        "SELECT count(*) FROM dam_station_link WHERE station_id NOT IN (SELECT station_id FROM wq_station)",
    "wq_station: not linked to any dam":
        "SELECT count(*) FROM wq_station WHERE station_id NOT IN (SELECT station_id FROM dam_station_link)",
    "dam_removal: HUC8 not in huc8_watershed":
        "SELECT count(*) FROM dam_removal WHERE HUC8 NOT IN (SELECT huc8 FROM huc8_watershed)",
    "dam_removal: NHDPlusID not in streams":
        "SELECT count(*) FROM dam_removal WHERE NHDPlusID NOT IN (SELECT nhdplus_id FROM streams)",
    "wq_station: NHDPlusID not in streams":
        "SELECT count(*) FROM wq_station WHERE NHDPlusID NOT IN (SELECT nhdplus_id FROM streams)",
    "wq_measurement: duplicate measurement_id":
        "SELECT count(*) - count(DISTINCT measurement_id) FROM wq_measurement",
    "wq_measurement: station_id not in wq_station":
        "SELECT count(*) FROM wq_measurement WHERE station_id NOT IN (SELECT station_id FROM wq_station)",
    "wq_measurement: value without standard unit":
        "SELECT count(*) FROM wq_measurement WHERE result_value_std IS NOT NULL AND result_unit_std IS NULL",
    "wq_measurement: null sample_date":
        "SELECT count(*) FROM wq_measurement WHERE sample_date IS NULL",
}

checks = pd.DataFrame(
    [(name, con.execute(sql).fetchone()[0]) for name, sql in CHECKS.items()],
    columns=["check", "violations"],
)
display(checks)
print("All checks pass" if checks["violations"].eq(0).all() else "SOME CHECKS FAILED")

## 3. Dams, stations and links

In [ ]:
dams = con.sql("""
    SELECT AR_ID, Dam_Name, River, State, HUC8, Year_Removed, Year_built,
           Dam_Height_ft, NHDPlusID, round(snap_distance_m, 1) AS snap_distance_m
    FROM dam_removal ORDER BY AR_ID
""").df()
dams

In [ ]:
# Linked stations per dam, by relationship to the dam.
con.sql("""
    SELECT AR_ID, relationship, count(*) AS stations,
           round(median(network_distance_km), 1) AS median_distance_km,
           round(max(network_distance_km), 1) AS max_distance_km
    FROM dam_station_link
    GROUP BY ALL ORDER BY AR_ID, relationship
""").df()

In [ ]:
# Snap distances: large values can mean a point snapped to the wrong reach.
snaps = con.sql("""
    SELECT 'dam' AS feature, snap_distance_m FROM dam_removal
    UNION ALL
    SELECT 'station', snap_distance_m FROM wq_station
""").df()
display(snaps.groupby("feature")["snap_distance_m"].describe().round(1))

print("Stations snapped more than 500 m away:")
con.sql("""
    SELECT station_id, station_name, round(snap_distance_m) AS snap_distance_m
    FROM wq_station WHERE snap_distance_m > 500 ORDER BY snap_distance_m DESC
""").df()

## 4. Map

HUC8 boundaries, the `streams` reaches, dams (black) and linked stations, colored by their relationship to the dam(s) they link to. A station linked to dams with different relationships is gray.

In [ ]:
dam_gdf = gpd.read_parquet(WAREHOUSE / "dam_removal.parquet")
station_gdf = gpd.read_parquet(WAREHOUSE / "wq_station.parquet")
huc8_gdf = gpd.read_parquet(WAREHOUSE / "huc8_watershed.parquet")
stream_gdf = gpd.read_parquet(WAREHOUSE / "streams.parquet")
links = pd.read_parquet(WAREHOUSE / "dam_station_link.parquet")

print(f"Streams to draw: {len(stream_gdf):,} (slow above ~20,000)")

RELATIONSHIP_COLORS = {"upstream": "orange", "downstream": "blue", "same_reach": "purple"}

station_links = (
    links.groupby("station_id")
    .agg(relationship=("relationship", lambda r: r.iloc[0] if r.nunique() == 1 else "mixed"),
         dams=("AR_ID", lambda d: ", ".join(sorted(d))))
    .reset_index()
)
station_map = station_gdf.merge(station_links, on="station_id")

center = huc8_gdf.to_crs(huc8_gdf.estimate_utm_crs()).union_all().centroid
center = gpd.GeoSeries([center], crs=huc8_gdf.estimate_utm_crs()).to_crs(4326).iloc[0]
m = folium.Map(location=[center.y, center.x], zoom_start=8, tiles="cartodbpositron")

folium.GeoJson(
    huc8_gdf[["huc8", "name", "geometry"]], name="HUC8",
    style_function=lambda _: {"color": "black", "weight": 2, "fillOpacity": 0.03},
    tooltip=folium.GeoJsonTooltip(["huc8", "name"]),
).add_to(m)

folium.GeoJson(
    stream_gdf[["nhdplus_id", "gnis_name", "geometry"]], name="Streams",
    style_function=lambda _: {"color": "#3a7bd5", "weight": 1, "opacity": 0.6},
    tooltip=folium.GeoJsonTooltip(["nhdplus_id", "gnis_name"]),
).add_to(m)

for row in station_map.itertuples():
    folium.CircleMarker(
        [row.geometry.y, row.geometry.x], radius=4, weight=1, fill=True, fill_opacity=0.8,
        color=RELATIONSHIP_COLORS.get(row.relationship, "gray"),
        tooltip=f"{row.station_id} ({row.relationship}) - dams: {row.dams}",
    ).add_to(m)

for row in dam_gdf.itertuples():
    folium.Marker(
        [row.geometry.y, row.geometry.x],
        icon=folium.Icon(color="black", icon="info-sign"),
        tooltip=f"{row.AR_ID}: {row.Dam_Name} (removed {row.Year_Removed})",
    ).add_to(m)

folium.LayerControl().add_to(m)
m

## 5. `wq_measurement`

In [ ]:
# Rows, stations, censoring and unit assumptions per parameter.
con.sql("""
    SELECT parameter_std, result_unit_std,
           count(*) AS n,
           count(DISTINCT station_id) AS stations,
           min(sample_date) AS first_date,
           max(sample_date) AS last_date,
           round(100 * avg(is_censored::INT), 1) AS pct_censored,
           round(100 * avg(unit_basis_assumed::INT), 1) AS pct_unit_assumed
    FROM wq_measurement
    GROUP BY ALL ORDER BY n DESC
""").df()

In [ ]:
# Rows by provider and fraction (nutrients keep their fraction; field parameters don't).
con.sql("""
    SELECT parameter_std, coalesce(sample_fraction_std, '(none)') AS fraction,
           provider, count(*) AS n
    FROM wq_measurement
    GROUP BY ALL ORDER BY parameter_std, fraction, provider
""").df()

In [ ]:
# Turbidity units are kept raw (NTU / NTRU / FNU are treated as comparable).
con.sql("""
    SELECT result_unit_raw, count(*) AS n
    FROM wq_measurement WHERE parameter_std = 'turbidity'
    GROUP BY ALL ORDER BY n DESC
""").df()

In [ ]:
values = con.sql("SELECT parameter_std, result_value_std FROM wq_measurement").df()
display(values.groupby("parameter_std")["result_value_std"].describe().round(3))

parameters = sorted(values["parameter_std"].unique())
ncols = 4
nrows = -(-len(parameters) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
for ax, parameter in zip(axes.flat, parameters):
    ax.hist(values.loc[values["parameter_std"] == parameter, "result_value_std"], bins=40)
    ax.set_title(f"{parameter} ({wq_rules.STANDARD_UNITS[parameter]})", fontsize=9)
for ax in axes.flat[len(parameters):]:
    ax.set_visible(False)
fig.tight_layout()

## 6. Raw vs cleaned row counts

Per WQP `CharacteristicName` in the 3.6.1 mapping: rows downloaded vs rows kept. The `clean_wq_results` Slurm log has the drop count for each rule.

In [ ]:
mapped = pd.DataFrame({"parameter": list(wq_rules.PARAMETERS)})
con.register("mapped_parameters", mapped)

con.sql("""
    WITH raw AS (
        SELECT CharacteristicName AS parameter, count(*) AS raw_rows
        FROM wq_measurement_raw
        WHERE CharacteristicName IN (SELECT parameter FROM mapped_parameters)
        GROUP BY ALL
    ),
    clean AS (
        SELECT parameter, count(*) AS kept_rows FROM wq_measurement GROUP BY ALL
    )
    SELECT parameter, raw_rows, coalesce(kept_rows, 0) AS kept_rows,
           round(100 * coalesce(kept_rows, 0) / raw_rows, 1) AS pct_kept
    FROM raw LEFT JOIN clean USING (parameter)
    ORDER BY raw_rows DESC
""").df()

## 7. Before/after coverage

A prototype of the step 4 `measurement_window` view: each measurement at a linked station is classified relative to its dam's removal year, using the same rule as `report_pilot_eligibility.py`:

- **pre**: the `years_before` calendar years before the removal year
- **post**: the `years_after` calendar years after it
- **removal_year**: excluded from both (only the year of removal is known)
- **outside**: beyond the window

A dam has a usable before/after comparison for a parameter only if **one station** has both pre and post data.

In [ ]:
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW measurement_window AS
    SELECT l.AR_ID, l.station_id, l.relationship, l.network_distance_km,
           d.Year_Removed AS removal_year,
           m.parameter_std, m.sample_fraction_std, m.sample_date,
           m.result_value_std, m.is_censored,
           CASE
               WHEN year(m.sample_date) - d.Year_Removed BETWEEN -{YEARS_BEFORE} AND -1 THEN 'pre'
               WHEN year(m.sample_date) - d.Year_Removed BETWEEN 1 AND {YEARS_AFTER} THEN 'post'
               WHEN year(m.sample_date) = d.Year_Removed THEN 'removal_year'
               ELSE 'outside'
           END AS period
    FROM dam_station_link l
    JOIN dam_removal d USING (AR_ID)
    JOIN wq_measurement m USING (station_id)
    WHERE d.Year_Removed IS NOT NULL
""")

# Measurements per dam and period.
con.sql("""
    SELECT AR_ID, removal_year, period, count(*) AS n
    FROM measurement_window
    GROUP BY ALL ORDER BY AR_ID, period
""").df().pivot_table(index=["AR_ID", "removal_year"], columns="period", values="n", fill_value=0)

In [ ]:
# Pairs (dam, station, parameter) with data on both sides of removal.
pair_coverage = con.sql("""
    SELECT AR_ID, station_id, relationship, parameter_std,
           count(*) FILTER (WHERE period = 'pre') AS n_pre,
           count(*) FILTER (WHERE period = 'post') AS n_post
    FROM measurement_window
    GROUP BY ALL
""").df()
pair_coverage["has_both"] = (pair_coverage["n_pre"] > 0) & (pair_coverage["n_post"] > 0)

# Dams x parameters: number of stations with both pre and post data.
(pair_coverage[pair_coverage["has_both"]]
 .pivot_table(index="AR_ID", columns="parameter_std", values="station_id",
              aggfunc="nunique", fill_value=0)
 .reindex(dams["AR_ID"], fill_value=0))

In [ ]:
# Same split by relationship: a downstream station with both is the main comparison.
(pair_coverage[pair_coverage["has_both"]]
 .groupby(["AR_ID", "relationship"])["station_id"].nunique()
 .unstack(fill_value=0)
 .reindex(dams["AR_ID"], fill_value=0))

In [ ]:
# Compare dam-level "has_both" with the existing eligibility report (raw, uncleaned
# results and verbatim characteristic names, so small differences are expected).
report_files = [
    ROOT / "data" / "processed" / f"pilot_eligibility_dams_{huc8}.csv" for huc8 in cfg.huc8s
]
report_files = [p for p in report_files if p.exists()]

if not report_files:
    print("No pilot_eligibility_dams_*.csv found; run report_pilot_eligibility to compare.")
else:
    old = pd.concat([pd.read_csv(p, dtype={"dam_id": str}) for p in report_files])
    old = old.groupby("dam_id")["has_both"].any().rename("has_both_report")
    new = pair_coverage.groupby("AR_ID")["has_both"].any().rename("has_both_warehouse")
    comparison = pd.concat([old, new], axis=1).fillna(False)
    comparison["agree"] = comparison["has_both_report"] == comparison["has_both_warehouse"]
    display(comparison)

## 8. Spot check: one dam's time series

Defaults to the dam and parameter with the most measurements inside the window. Change `DAM_ID` / `PARAMETER` to look at others.

In [ ]:
busiest = con.sql("""
    SELECT AR_ID, parameter_std, count(*) AS n
    FROM measurement_window WHERE period IN ('pre', 'post')
    GROUP BY ALL ORDER BY n DESC LIMIT 1
""").df()
display(busiest)

DAM_ID = busiest["AR_ID"].iloc[0] if len(busiest) else None
PARAMETER = busiest["parameter_std"].iloc[0] if len(busiest) else None

In [ ]:
series = con.execute("""
    SELECT station_id, relationship, sample_date, result_value_std, is_censored, removal_year
    FROM measurement_window
    WHERE AR_ID = ? AND parameter_std = ?
    ORDER BY sample_date
""", [DAM_ID, PARAMETER]).df()

if series.empty:
    print("No measurements for this dam and parameter.")
else:
    removal_year = int(series["removal_year"].iloc[0])
    fig, ax = plt.subplots(figsize=(12, 4))
    for (station, relationship), group in series.groupby(["station_id", "relationship"]):
        ax.plot(pd.to_datetime(group["sample_date"]), group["result_value_std"],
                marker="o", markersize=3, linewidth=0.8, label=f"{station} ({relationship})")
    ax.axvspan(pd.Timestamp(removal_year, 1, 1), pd.Timestamp(removal_year, 12, 31),
               color="red", alpha=0.15, label="removal year (excluded)")
    ax.axvspan(pd.Timestamp(removal_year - YEARS_BEFORE, 1, 1), pd.Timestamp(removal_year - 1, 12, 31),
               color="gray", alpha=0.08, label="pre window")
    ax.axvspan(pd.Timestamp(removal_year + 1, 1, 1), pd.Timestamp(removal_year + YEARS_AFTER, 12, 31),
               color="green", alpha=0.08, label="post window")
    ax.set_title(f"{DAM_ID}: {PARAMETER} ({wq_rules.STANDARD_UNITS[PARAMETER]})")
    ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1))
    fig.tight_layout()

## Findings

_Notes from running this notebook: anything that looks wrong, and what to change before the 30-50 dam run._

-